# Greek Protipa Exams Dataset: EDA & Samples

This notebook demonstrates how to load the **Greek Protipa Exams** dataset from Hugging Face, inspect samples, and perform Exploratory Data Analysis (EDA) using specialized utility functions.

In [34]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
import sys
import os
sys.path.append(os.path.abspath('../src'))
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from protipa_exams_dataset.eda_utils import display_qa
from dotenv import load_dotenv, find_dotenv
import numpy as np
import random

from protipa_exams_dataset.eda_utils import (
    print_sample, 
    display_samples,
    display_multimodal_samples, 
    plot_subject_dist,      
    plot_level_dist,        
    plot_format_dist,       
    plot_reference_dist,    
    plot_temporal_distribution,
    plot_points_distribution,
    print_source_files,
)

# Load environment variables
load_dotenv(find_dotenv())

repo_id = os.getenv("HF_REPO_ID")
token = os.getenv("HF_TOKEN")

print(f"Using Repository: {repo_id}")



Using Repository: ilsp/greek-protipa-exams


## 1. Load the Dataset
We load the `test` split, which contains the consolidated data.

In [36]:
dataset = load_dataset(repo_id, split="test", token=token, download_mode="force_redownload")
df = dataset.to_pandas()

print(f"✅ Dataset loaded. Total rows: {len(df)}")

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1646 [00:00<?, ? examples/s]

✅ Dataset loaded. Total rows: 1646


## 2. Main Samples Inspection
We use the `display_samples` utility to look at a few random questions in detail.

In [81]:
science_df = df[df['subject'].isin(['mathematics', 'physics'])]
all_science_ids = science_df['id'].tolist()
random_id = random.choice(all_science_ids)
display_qa(df, random_id)
print_source_files(df, sample_id=random_id)

🔍 Identifying source files for ID: math_lyc_2021_1_14
📂 Searching in: /data/home/prokopis/src/protipa-exams-dataset/data
--------------------------------------------------


✅ **Source JSON:** [../data/2021/ΛΥΚΕΙΟ/ΜΑΘΗΜΑΤΙΚΑ/ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2021_1.json](../data/2021/ΛΥΚΕΙΟ/ΜΑΘΗΜΑΤΙΚΑ/ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2021_1.json)

✅ **Source MD:** &nbsp;&nbsp; [../data/2021/ΛΥΚΕΙΟ/ΜΑΘΗΜΑΤΙΚΑ/ΑΠΑΝΤΗΣΕΙΣ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2021_1.md](../data/2021/ΛΥΚΕΙΟ/ΜΑΘΗΜΑΤΙΚΑ/ΑΠΑΝΤΗΣΕΙΣ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2021_1.md)

--------------------------------------------------


In [56]:
test_id = "math_lyc_2016_1_7"
display_qa(df, test_id)
print_source_files(df, sample_id=test_id)


🔍 Identifying source files for ID: math_lyc_2016_1_7
📂 Searching in: /data/home/prokopis/src/protipa-exams-dataset/data
--------------------------------------------------


✅ **Source JSON:** [../data/2016/ΛΥΚΕΙΟ/ΜΑΘΗΜΑΤΙΚΑ/ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2016_1.json](../data/2016/ΛΥΚΕΙΟ/ΜΑΘΗΜΑΤΙΚΑ/ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2016_1.json)

✅ **Source MD:** &nbsp;&nbsp; [../data/2016/ΛΥΚΕΙΟ/ΜΑΘΗΜΑΤΙΚΑ/ΑΠΑΝΤΗΣΕΙΣ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2016_1.md](../data/2016/ΛΥΚΕΙΟ/ΜΑΘΗΜΑΤΙΚΑ/ΑΠΑΝΤΗΣΕΙΣ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2016_1.md)

--------------------------------------------------


## 3. Multimodal Analysis (Samples with Images)
Let's specifically look at questions that include visual aids. We display them as a DataFrame first, then show the images.

In [ ]:
display_samples(df, n=3, prioritize_images=True)

## 4. Exploratory Data Analysis (EDA)

### 4.1 Distribution by Subject and Admission Level

In [ ]:

# Subject Distribution
fig_subj, df_subj = plot_subject_dist(df)
display(df_subj)
display(fig_subj)

In [ ]:
# Admission Level Distribution
fig_level, df_level = plot_level_dist(df)
display(df_level)
display(fig_level)

### 4.2 Distribution by Format and Reference Type

In [ ]:
# Question Format Distribution
fig_format, df_format = plot_format_dist(df)
display(df_format)
display(fig_format)

In [ ]:
# Reference Type Distribution
fig_ref, df_ref = plot_reference_dist(df)
display(df_ref)
display(fig_ref)

### 4.3 Temporal Trends
How are questions distributed over the years per subject?

In [ ]:
fig_temp, df_temp = plot_temporal_distribution(df)
display(df_temp)
display(fig_temp)

### 4.4 Point Values Analysis

In [ ]:
fig_points, df_points = plot_points_distribution(df)
display(df_points)
display(fig_points)

In [ ]:
from protipa_exams_dataset.eda_utils import plot_format_by_subject
# Returns the matplotlib figure and the pivot table of counts
fig_fmt, df_fmt = plot_format_by_subject(df)
display(fig_fmt)
display(df_fmt)

### 4.5 Unique verification

In [ ]:

# Create a temporary series where 'choices' are tuples so they can be hashed/compared
choices_hashable = df['choices'].apply(lambda x: tuple(x) if isinstance(x, (list, np.ndarray)) else x)

# 1. Total duplicates (Metadata + Choices)
# We find all standard hashable columns and manually include the converted choices
metadata_cols = [col for col in df.columns if df[col].apply(lambda x: not isinstance(x, (list, np.ndarray))).all()]
check_subset = metadata_cols + (['choices'] if 'choices' in df.columns else [])

# We use assign to temporarily hold the hashable choices for the check
total_duplicates = df.assign(choices=choices_hashable).duplicated(subset=check_subset).sum()
print(f"Total duplicate rows (Metadata + Choices): {total_duplicates}")

# 2. Check uniqueness of the 'id' column
if 'id' in df.columns:
    is_id_unique = df['id'].is_unique
    print(f"Is 'id' column unique? {is_id_unique}")
    if not is_id_unique:
        num_duplicates = df.duplicated('id').sum()
        print(f"Number of duplicate IDs: {num_duplicates}")


# 3. Check for rows with identical [Question + Choices + Input]
# This defines a "Semantic Duplicate" (same content, regardless of ID/Year)
subset_to_check = []
if 'question' in df.columns: subset_to_check.append('question')
if 'input' in df.columns:    subset_to_check.append('input')
subset_to_check.append('c_h') # This is our tuple-version of choices

content_duplicates = df.assign(c_h=choices_hashable).duplicated(subset=subset_to_check).sum()

print(f"Rows with identical content ({' + '.join([c if c!='c_h' else 'choices' for c in subset_to_check])}): {content_duplicates}")

# Optional: To actually see them
if content_duplicates > 0:
    print("\nDisplaying samples of content duplicates:")
    dup_rows = df.assign(c_h=choices_hashable)[df.assign(c_h=choices_hashable).duplicated(subset=subset_to_check, keep=False)]
    display(dup_rows.sort_values(by='question')[['id', 'subject', 'year', 'question', 'input']])

# 4. Check for nulls in critical columns
critical_cols = ['id', 'subject', 'admission_level', 'question']
existing_critical = [c for c in critical_cols if c in df.columns]
null_counts = df[existing_critical].isnull().sum()

if null_counts.sum() > 0:
    print("\nNull values in critical columns:")
    print(null_counts[null_counts > 0])
else:
    print(f"\n✅ No null values in critical columns ({', '.join(existing_critical)}).")

In [ ]:
# Filter for duplicate questions
duplicate_questions_df = df[df.duplicated(subset=['question'], keep=False)].sort_values(by='question')

# Display with specific CSS for wrapping and column width
(duplicate_questions_df[['id', 'subject', 'year', 'question']]
 .style.set_properties(**{
     'text-align': 'left',
     'white-space': 'pre-wrap',  # Correctly handles newlines and wrapping
     'max-width': '600px',       # Limits width so it doesn't stretch the screen
     'vertical-align': 'top'
 }))

In [ ]:
# 1. Find rows with literal \n (legacy escaped newlines)
literal_nl_mask = df['answer_text'].str.contains(r'\\n', regex=True, na=False)
literal_nl_ids = df[literal_nl_mask]['id'].tolist()
# 2. Find rows with stringified JSON lists ["...", "..."]
json_list_mask = df['answer_text'].str.startswith('[', na=False) & df['answer_text'].str.endswith(']', na=False)
json_list_ids = df[json_list_mask]['id'].tolist()
# 3. Find "True" multi-line answers (already using real newlines)
true_multiline_mask = df['answer_text'].str.contains('\n', na=False)
true_multiline_ids = df[true_multiline_mask]['id'].tolist()
print(f"📊 Rows with literal '\\n': {len(literal_nl_ids)}")
print(f"IDs: {literal_nl_ids[:10]} ...\n")
print(f"📦 Rows with stringified lists: {len(json_list_ids)}")
print(f"IDs: {json_list_ids[:10]} ...\n")
print(f"✅ Rows already using real newlines: {len(true_multiline_ids)}")
print(f"IDs: {true_multiline_ids[:10]} ...")

In [ ]:
from IPython.display import display, Markdown
import random

dataset = load_dataset(repo_id, split="test", token=token, download_mode="force_redownload")

# Select specific ID
target_id = "math_lyc_2025_2_31"
target_id = "math_lyc_2013_2_6.3"
target_id = "math_lyc_2016_1_10.2"
target_id = "math_lyc_2016_2_9.2"

subset = dataset.filter(lambda x: x["id"] == target_id)

if len(subset) > 0:
    sample = subset[0]
    
    # Construct Markdown output
    md_output = f"### ID: {sample['id']}\n"
    md_output += f"**Question:** {sample['question']}\n\n"

    # 1. Handle Multimodal Metadata
    if sample.get('image_description'):
        md_output += f"🖼️ **Image Description:** {sample['image_description']}\n\n"

    if sample.get('image_transcription'):
        md_output += f"📝 **Image Transcription:** {sample['image_transcription']}\n\n"

    # 2. Handle Choices
    if sample.get('choices'):
        md_output += "#### Choices:\n"
        correct_idx = sample.get('answer_index')
        for i, choice in enumerate(sample['choices']):
            marker = "✅" if i == correct_idx else "&nbsp;&nbsp;&nbsp;"
            md_output += f"- [{marker}] **{i}:** {choice}\n"
        md_output += f"\n**Correct Answer Index:** `{correct_idx}`"
    else:
        md_output += f"\n**Answer:** {sample['answer_text']}"

    # Render everything
    display(Markdown(md_output))
else:
    print(f"ID {target_id} not found in dataset.")